# AI应用基础 · 工单M2.2《轴承寿命预测系统》
**第7周 | 2课时(80分钟) | 理实一体**  | 学生实训版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 对比LSTM与线性基线的误差曲线，指出LSTM在哪个退化阶段增益最大；示范讲解 |
| **B 组长** | 标准达成 | MSE<0.10；对比≥2模型；解读有业务意义 |
| **C 操作员** | 基础完成 | MSE<0.15；能预测与可视化 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）


## 一、课前资讯（Predict·预测）
> **预习快问**：窗口长度20、步长1，长度200的序列能切出多少个样本？

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m2rul"], check=True)
print("数据已生成: data/")

## 三、任务要求（Make）
**实施步骤**
1. 可视化一个“输入窗口→标签RUL”样例
2. 训练RULNet(≤60 epochs)
3. 计算MSE/RMSE/MAE
4. 绘制真实vs预测RUL曲线+业务解读

**验收标准**：及格(D)=`MSE<0.15；能预测与可视化` ；优秀(B+)=`MSE<0.10；对比≥2模型；解读有业务意义`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
# -*- coding: utf-8 -*-
"""M2.2 轴承剩余寿命(RUL)预测(LSTM, CPU) —— 学生代码框架"""
import numpy as np
import torch
import torch.nn as nn

DATA = "data/bearing_rul.npz"

### Starter 2：`class RULNet(nn.Module):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
class RULNet(nn.Module):

    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden,
                            batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):                # x: (N, 20, 1)
        out, _ = self.lstm(x)            # (N,20,H)
        last = out[:, -1, :]             # 取最后时刻 → (N,H)
        return self.fc(last).squeeze(-1)  # (N,)

### Starter 3：`def visualize_window():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def visualize_window():
    """预习用: 看见'输入窗口→标签'的对应关系(运行即出图)"""
    import matplotlib.pyplot as plt
    d = np.load(DATA)
    X, y = d["X"], d["y"]
    i = 100
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(range(i, i + 20), X[i], "o-", label="输入窗口(振动)")
    ax.axvline(i + 20, color="r", ls="--", label=f"预测点RUL={y[i]:.2f}")
    ax.set_xlabel("时间步")
    ax.legend()
    plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS"]
    plt.tight_layout()
    plt.savefig("window_demo.png", dpi=120)
    print("已保存 window_demo.png")

### Starter 4：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def main():
    d = np.load(DATA)
    X, y = d["X"].astype(np.float32), d["y"].astype(np.float32)
    X = X[:, :, None]                    # (N,20)→(N,20,1)
    n = int(len(X) * 0.8)
    Xtr, ytr = torch.tensor(X[:n]), torch.tensor(y[:n])
    Xte, yte = torch.tensor(X[n:]), torch.tensor(y[n:])

    model = RULNet()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    lossf = nn.MSELoss()

    for epoch in range(60):
        model.train()
        # TODO-1: 全批量(或batch=64)训练, 每10轮打印loss
        # === 你的代码 ===
        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                mse = lossf(model(Xte), yte).item()
            print(f"epoch {epoch + 1}  验证MSE={mse:.4f}")

    # TODO-2: 计算测试集 MSE/RMSE/MAE 三项指标
    # TODO-3: 选1台设备的退化段, 绘制"真实RUL vs 预测RUL"曲线
    # TODO-4: (对照)线性回归基线: 用窗口均值单特征拟合, 对比MSE
    # TODO-5: (业务解读)≤80字: RMSE=0.1对维护计划意味着什么

### Starter 5：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
if __name__ == "__main__":
    import sys
    if "--viz" in sys.argv:
        visualize_window()
    else:
        main()

## 四、提交要求与避坑
- 提交: `学号_姓名_M2.2.py` 或本notebook(含输出)
- ⚠ 窗口X与标签y错位是头号bug，用--viz可视化核对
- ⚠ CPU训练加早停，epochs≤60观察验证loss
- ⚠ RUL是回归任务，损失用MSELoss而非交叉熵